In [2]:
import pandas as pd
import re
from pathlib import Path

# =========================
# 1. 파일 경로 설정
# =========================
input_file = r"서울교통공사_지하철혼잡도정보_통합본.csv"
output_dir = Path("erd_output")
output_dir.mkdir(exist_ok=True)

# =========================
# 2. 원본 데이터 읽기
# =========================
df = pd.read_csv(input_file, encoding="utf-8-sig")

# 컬럼명 공백 제거
df.columns = df.columns.str.strip()

# =========================
# 3. 시간대 컬럼 찾기
#    예: 5시30분, 6시00분 ...
# =========================
time_pattern = re.compile(r"^\d{1,2}시\d{2}분$")
time_cols = [col for col in df.columns if time_pattern.match(col)]

# 고정 컬럼
base_cols = ["요일구분", "호선", "역번호", "출발역", "상하구분", "기준일자", "기준연도", "기준월"]

# 누락 확인
missing_cols = [c for c in base_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"다음 필수 컬럼이 없습니다: {missing_cols}")

print("시간대 컬럼 개수:", len(time_cols))
print("시간대 컬럼 예시:", time_cols[:5])

# =========================
# 4. line_info 생성
# =========================
line_info = (
    df[["호선"]]
    .drop_duplicates()
    .sort_values("호선")
    .reset_index(drop=True)
)
line_info["line_id"] = range(1, len(line_info) + 1)
line_info = line_info[["line_id", "호선"]].rename(columns={"호선": "line_name"})

# =========================
# 5. station_info 생성
# =========================
station_info = (
    df[["역번호", "출발역", "호선"]]
    .drop_duplicates()
    .merge(line_info, left_on="호선", right_on="line_name", how="left")
    .sort_values(["line_id", "역번호", "출발역"])
    .reset_index(drop=True)
)

station_info["station_id"] = range(1, len(station_info) + 1)

station_info = station_info[["station_id", "역번호", "출발역", "line_id"]].rename(
    columns={
        "역번호": "station_no",
        "출발역": "station_name"
    }
)

# =========================
# 6. date_info 생성
# =========================
date_info = (
    df[["기준일자", "기준연도", "기준월", "요일구분"]]
    .drop_duplicates()
    .sort_values(["기준일자", "요일구분"])
    .reset_index(drop=True)
)

date_info["date_id"] = range(1, len(date_info) + 1)

date_info = date_info[["date_id", "기준일자", "기준연도", "기준월", "요일구분"]].rename(
    columns={
        "기준일자": "base_date",
        "기준연도": "base_year",
        "기준월": "base_month",
        "요일구분": "day_type"
    }
)

# =========================
# 7. time_slot_info 생성
# =========================
def to_24h(label):
    # 예: "5시30분" -> "05:30"
    m = re.match(r"^(\d{1,2})시(\d{2})분$", label)
    hh = int(m.group(1))
    mm = m.group(2)
    return f"{hh:02d}:{mm}"

time_slot_info = pd.DataFrame({
    "time_label": time_cols,
    "time_order": range(1, len(time_cols) + 1),
    "time_24h": [to_24h(x) for x in time_cols]
})

time_slot_info["time_slot_id"] = range(1, len(time_slot_info) + 1)
time_slot_info = time_slot_info[["time_slot_id", "time_label", "time_order", "time_24h"]]

# =========================
# 8. station_key / date_key 붙이기
# =========================
df2 = df.merge(
    line_info,
    left_on="호선",
    right_on="line_name",
    how="left"
)

df2 = df2.merge(
    station_info,
    left_on=["역번호", "출발역", "line_id"],
    right_on=["station_no", "station_name", "line_id"],
    how="left"
)

df2 = df2.merge(
    date_info,
    left_on=["기준일자", "기준연도", "기준월", "요일구분"],
    right_on=["base_date", "base_year", "base_month", "day_type"],
    how="left"
)

# =========================
# 9. 가로형 -> 세로형 변환
# =========================
fact_long = df2.melt(
    id_vars=["station_id", "date_id", "상하구분"],
    value_vars=time_cols,
    var_name="time_label",
    value_name="congestion_value"
)

# 시간대 키 연결
fact_long = fact_long.merge(
    time_slot_info,
    on="time_label",
    how="left"
)

# 최종 fact 테이블
congestion_fact = fact_long[[
    "station_id",
    "date_id",
    "time_slot_id",
    "상하구분",
    "congestion_value"
]].rename(columns={"상하구분": "direction"})

# 결측 제거
congestion_fact = congestion_fact.dropna(subset=["station_id", "date_id", "time_slot_id", "congestion_value"])

# 타입 정리
congestion_fact["station_id"] = congestion_fact["station_id"].astype(int)
congestion_fact["date_id"] = congestion_fact["date_id"].astype(int)
congestion_fact["time_slot_id"] = congestion_fact["time_slot_id"].astype(int)
congestion_fact["congestion_value"] = pd.to_numeric(congestion_fact["congestion_value"], errors="coerce")

congestion_fact = congestion_fact.dropna(subset=["congestion_value"]).reset_index(drop=True)

# fact용 surrogate key
congestion_fact["congestion_id"] = range(1, len(congestion_fact) + 1)
congestion_fact = congestion_fact[[
    "congestion_id",
    "station_id",
    "date_id",
    "time_slot_id",
    "direction",
    "congestion_value"
]]

# =========================
# 10. CSV 저장
# =========================
line_info.to_csv(output_dir / "line_info.csv", index=False, encoding="cp949")
station_info.to_csv(output_dir / "station_info.csv", index=False, encoding="cp949")
date_info.to_csv(output_dir / "date_info.csv", index=False, encoding="cp949")
time_slot_info.to_csv(output_dir / "time_slot_info.csv", index=False, encoding="cp949")
congestion_fact.to_csv(output_dir / "congestion_fact.csv", index=False, encoding="cp949")

print("저장 완료")
print(line_info.shape, "line_info")
print(station_info.shape, "station_info")
print(date_info.shape, "date_info")
print(time_slot_info.shape, "time_slot_info")
print(congestion_fact.shape, "congestion_fact")

시간대 컬럼 개수: 39
시간대 컬럼 예시: ['5시30분', '6시00분', '6시30분', '7시00분', '7시30분']
저장 완료
(8, 2) line_info
(288, 4) station_info
(24, 5) date_info
(39, 4) time_slot_info
(517886, 6) congestion_fact
